In [49]:
words = open('names.txt', 'r').read().splitlines()

In [50]:
import torch
import torch.nn.functional as F

In [51]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [52]:
# create dataset
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print(f'Number of examples: ', num)

# initialize 'network'
g = torch.Generator().manual_seed(2147483648)
w = torch.randn((27, 27), generator=g, requires_grad=True)

Number of examples:  228146


In [53]:
# gradient decent
for k in range(100):

  # forward pass
  logits = w[xs] # log-counts
  counts = logits.exp() # equivalent to N in the previous model
  probs = counts / counts.sum(1, keepdim=True) # normalizing each row
  loss = -(probs[torch.arange(num), ys]).log().mean() + 0.1*(w**2).mean()
  print(loss.item())

  # backward pass
  w.grad = None
  loss.backward()

  # update
  w.data += -50 * w.grad

3.7973990440368652
3.3985543251037598
3.193441390991211
3.0632364749908447
2.9705841541290283
2.9032697677612305
2.853576183319092
2.8160133361816406
2.7867627143859863
2.763288736343384
2.743964910507202
2.7277424335479736
2.713918924331665
2.7020034790039062
2.6916308403015137
2.682525634765625
2.6744754314422607
2.6673147678375244
2.6609129905700684
2.655165672302246
2.6499874591827393
2.645308494567871
2.6410675048828125
2.637213945388794
2.6337039470672607
2.6304988861083984
2.627565860748291
2.6248762607574463
2.6224043369293213
2.6201281547546387
2.618028402328491
2.616088628768921
2.614292621612549
2.612628221511841
2.611083507537842
2.609647274017334
2.608311414718628
2.607067108154297
2.6059064865112305
2.604823350906372
2.6038105487823486
2.6028642654418945
2.601977825164795
2.6011478900909424
2.600369453430176
2.5996391773223877
2.5989537239074707
2.5983095169067383
2.597703456878662
2.5971333980560303
2.5965964794158936
2.596090316772461
2.595613479614258
2.595163583755493

In [54]:
# sampling
for _ in range(5):
  word = []
  ix = 0
  while True:

    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = w[ix]
    counts = logits.exp()
    p = counts / counts.sum(dim=0, keepdim=True)

    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()

    word.append(itos[ix])
    if ix == 0:
      break

  print(''.join(word))

eoseleoukagamevia.
ilyjxq.
zari.
chakahusliciakaryr.
weislra.
